In [1]:
import os

def show_inputs(depth=4, max_files_per_dir=6):
    """Print what is attached under /kaggle/input - check it before trusting the rest."""
    root0 = os.environ.get("SAT_INPUT", "/kaggle/input")
    for root, dirs, files in os.walk(root0):
        dirs[:] = sorted(d for d in dirs if not d.startswith("."))
        level = root.replace(root0, "").count(os.sep)
        if level > depth:
            dirs[:] = []
            continue
        print("  " * level + root + "/")
        for fn in sorted(files)[:max_files_per_dir]:
            print("  " * level + "  " + fn)
        if len(files) > max_files_per_dir:
            print("  " * level + f"  ... +{len(files) - max_files_per_dir} more files")

show_inputs()

/kaggle/input/
  /kaggle/input/datasets/
    /kaggle/input/datasets/rifat4018/
      /kaggle/input/datasets/rifat4018/sat-ext/
        /kaggle/input/datasets/rifat4018/sat-ext/sat_ext/
          README.md
          analysis.py
          audit.py
          config.py
          dataset.py
          dota.py
          ... +10 more files
      /kaggle/input/datasets/rifat4018/sat-splits/
        /kaggle/input/datasets/rifat4018/sat-splits/audit/
        /kaggle/input/datasets/rifat4018/sat-splits/splits/
      /kaggle/input/datasets/rifat4018/satelite-coco/
        README.dataset.txt
        README.roboflow.txt
        /kaggle/input/datasets/rifat4018/satelite-coco/test/
          1011_jpeg.rf.141ad0f26ab02f1bfee50847febecb85.jpg
          1016_jpeg.rf.0aef8e3664a12b1681da06035362ed54.jpg
          1025_jpeg.rf.ce6b95f0170aab49e8aab8728fca22a8.jpg
          1067_jpeg.rf.b20f17d8c4416f8b8ed687b14a48966a.jpg
          1077_jpeg.rf.0e1e2f1b96f4487629a56a6b93755174.jpg
          1087_jpeg.rf.432

In [2]:
import os, sys, warnings
warnings.filterwarnings("ignore")
INPUT_ROOT = os.environ.get("SAT_INPUT", "/kaggle/input")

def _find_code_root():
    for root, dirs, files in os.walk(INPUT_ROOT):
        if {"config.py", "audit.py", "metrics.py", "infer.py"} <= set(files):
            return root
    return None

CODE_ROOT = os.environ.get("SAT_CODE") or _find_code_root()
assert CODE_ROOT, "sat_ext code folder not found - Add Input -> your 'sat-ext' code Dataset"
sys.path.insert(0, CODE_ROOT)
print("CODE_ROOT:", CODE_ROOT)

import numpy as np, pandas as pd
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 60); pd.set_option("display.max_colwidth", 70)
import config as cfg, utils
utils.ensure_dirs(); utils.set_seed(cfg.SEED)
print("work dir:", cfg.WORK_DIR, "| smoke test:", cfg.SMOKE)

CODE_ROOT: /kaggle/input/datasets/rifat4018/sat-ext/sat_ext
work dir: /kaggle/working | smoke test: False


In [3]:
utils.warn_if_gpu_wasted()
import dataset as ds, extra_splits as xs
sdir, base, meta, bc = xs.load_base("grouped")
raw = ds.load_raw(ds.find_coco_root("main"))
cmap = ds.category_map(raw)
imgs = ds.image_table(raw)
boxes, log = ds.clean_annotations(raw, cmap)
id_map = dict(zip(base.key, base.image_id))
made = {}
made["grouped_1copy"] = ds.write_split("grouped_1copy", xs.one_copy(base, bc), imgs, boxes, log=log, id_map=id_map,
                                       extra=dict(dataset="main", base=meta["fingerprint"], variant="1 augmented copy per train source"))
for f in cfg.LEARNING_CURVE:
    name = f"grouped_lc{int(f * 100)}"
    made[name] = ds.write_split(name, xs.learning_curve(base, f), imgs, boxes, log=log, id_map=id_map,
                                extra=dict(dataset="main", base=meta["fingerprint"], variant=f"{int(f * 100)}% of train groups"))
for name in made:      # val/test must be identical to the base split
    for sp in ("val", "test"):
        a = utils.load_json(os.path.join(sdir, "coco", f"{sp}.json"))
        b = utils.load_json(os.path.join(cfg.SPLITS_OUT, name, "coco", f"{sp}.json"))
        assert a["images"] == b["images"] and len(a["annotations"]) == len(b["annotations"]), (name, sp)
print("val/test identical to 'grouped' in every ablation split: OK")

base 'grouped' (f7cee44b1365) from /kaggle/input/datasets/rifat4018/sat-splits/splits/grouped; audit table: /kaggle/input/datasets/rifat4018/sat-splits/splits/audit_main/05_images_groups.csv
split 'grouped_1copy' fingerprint b9e10f51802e
       images  groups  annotations  empty_images  plane  vehicle  ship
train     382     372        13250            10   1944    10185  1121
val        80      80         2887             2    433     2228   226
test       82      80         2735             0    433     2092   210
split 'grouped_lc25' fingerprint 57c982f32bf2
       images  groups  annotations  empty_images  plane  vehicle  ship
train     248      93         7671             4   1453     5541   677
val        80      80         2887             2    433     2228   226
test       82      80         2735             0    433     2092   210
split 'grouped_lc50' fingerprint 950f5ff9ca49
       images  groups  annotations  empty_images  plane  vehicle  ship
train     497     186        17

In [4]:
try:
    NAT = ds.find_coco_root("native")
except FileNotFoundError as e:
    NAT = None
    print("native export not attached -> native splits skipped.", e)
if NAT:
    nraw = ds.load_raw(NAT)
    nimgs = ds.image_table(nraw)
    nboxes, nlog = ds.clean_annotations(nraw, ds.category_map(nraw))
    print("native images:", len(nimgs), "| sizes (top 5):", nimgs.groupby(["width", "height"]).size().nlargest(5).to_dict())
    nman = xs.native_manifest(base, nimgs)
    m1 = ds.write_split("native_grouped", nman, nimgs, nboxes, log=nlog,
                        extra=dict(dataset="native", base=meta["fingerprint"], variant="native resolution"))
    nid = pd.read_csv(os.path.join(cfg.SPLITS_OUT, "native_grouped", "manifest.csv"))
    tdir = os.path.join(cfg.WORK_DIR, "_tiles")
    trows, tboxes = xs.make_tiles(nimgs, nboxes, set(nman[nman.split == "train"].key), tdir)
    grp = dict(zip(nman.key, nman.group))
    tman = pd.concat([nman[nman.split != "train"],
                      pd.DataFrame(dict(key=trows.key, split="train", group=trows.parent.map(grp)))])
    m2 = ds.write_split("native_tiles", tman, pd.concat([nimgs, trows.drop(columns=["parent"])], ignore_index=True),
                        {**nboxes, **tboxes}, log=nlog, id_map=dict(zip(nid.key, nid.image_id)), tiles_from=tdir,
                        extra=dict(dataset="native", base=meta["fingerprint"], variant=f"{cfg.TILE}px train tiles"))
    import shutil; shutil.rmtree(tdir, ignore_errors=True)
print("splits written:", sorted(os.listdir(cfg.SPLITS_OUT)))

native images: 536 | sizes (top 5): {(1920, 1080): 522, (2048, 1152): 14}
split 'native_grouped' fingerprint 2c526ef767ce
       images  groups  annotations  empty_images  plane  vehicle  ship
train     374     364        13816             0   1944    10672  1200
val        80      80         3105             0    433     2474   198
test       82      80         2817             0    434     2174   209
tiles: {'tiles': 2992, 'empty_tiles': 944, 'boxes_cut_below_min_visibility': 3990}
split 'native_tiles' fingerprint 40ad0ef2ec1e
       images  groups  annotations  empty_images  plane  vehicle  ship
train    2992     364        21983           944   3142    16948  1893
val        80      80         3105             0    433     2474   198
test       82      80         2817             0    434     2174   209
splits written: ['grouped_1copy', 'grouped_lc25', 'grouped_lc50', 'native_grouped', 'native_tiles']
